<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# Procesamiento de lenguaje natural

## Custom embedddings con Gensim


### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)


In [ ]:
import pandas
import os
from gensim.models import Word2Vec

**Usaré `La Odisea - Homero` para esta consigna.**


In [ ]:
if not os.path.exists("the_odyssey.txt"):
    !curl -L -o the_odyssey.txt "https://www.gutenberg.org/cache/epub/1727/pg1727.txt"
    print("Descarga completa")
else:
    print("El archivo ya se encuentra descargado")

In [ ]:
with open("the_odyssey.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Cantidad de caracteres (raw): {len(raw_text):,}")

### Limpieza del texto

Los archivos de Project Gutenberg incluyen un encabezado y un pie de página con información legal/administrativa (licencia, título, idioma, etc.) que no forma parte del texto original. Los removemos delimitando el contenido entre los marcadores `*** START OF ... ***` y `*** END OF ... ***`.


In [ ]:
import re


def strip_gutenberg_metadata(text: str) -> str:
    start_match = re.search(r"\*\*\* START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)
    end_match = re.search(r"\*\*\* END OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)

    start_idx = start_match.end() if start_match else 0
    end_idx = end_match.start() if end_match else len(text)

    return text[start_idx:end_idx].strip()


text = strip_gutenberg_metadata(raw_text)
print(f"Cantidad de caracteres (raw):        {len(raw_text):,}")
print(f"Cantidad de caracteres (sin metadata): {len(text):,}")

In [ ]:
# Verificamos que el recorte haya quedado bien (debería empezar/terminar en el texto, no en boilerplate)
print("--- INICIO ---")
print(text[:300])
print("\n--- FIN ---")
print(text[-300:])

El texto del "FIN" parece que son footnotes con información extra o notas de los editores respecto al texto. Las dejamos ya que pueden enriquecer el contenido.


### Normalización y tokenización

Pasamos el texto a minúsculas, quitamos puntuación/caracteres no alfabéticos y tokenizamos por oración, que es el formato que espera `Word2Vec` de Gensim (lista de oraciones, cada una lista de tokens).


In [ ]:
from gensim.utils import simple_preprocess


raw_sentences = re.split(r"(?<=[.!?])\s+", text)

sentences = [simple_preprocess(sentence, deacc=True) for sentence in raw_sentences]
sentences = [s for s in sentences if len(s) > 0]

print(f"Cantidad de oraciones: {len(sentences):,}")
print(f"Ejemplo de oración tokenizada: {sentences[0]}")

## Entrenamiento de embeddings con Word2Vec

Usamos la arquitectura Skipgram (`sg=1`), que suele funcionar mejor que CBOW con corpus pequeños como este (~3.500 oraciones). Sobrecargamos el callback de Gensim para poder ver el loss por época durante el entrenamiento.


In [ ]:
from gensim.models.callbacks import CallbackAny2Vec


class LossLogger(CallbackAny2Vec):
    """Imprime el loss de cada época (Gensim solo trackea el acumulado)."""

    def __init__(self):
        self.epoch = 0
        self.loss_previous_step = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print(f"Loss after epoch {self.epoch}: {loss}")
        else:
            print(f"Loss after epoch {self.epoch}: {loss - self.loss_previous_step}")
        self.epoch += 1
        self.loss_previous_step = loss

**Justificación de los hiperparámetros**

- **`sg=1` (Skipgram):** Skipgram predice el contexto a partir de la palabra objetivo, lo que genera más actualizaciones de gradiente por ocurrencia de cada palabra (una por cada palabra del contexto) en vez de una sola por ventana. Con pocos datos, eso le da mejores representaciones a las palabras poco frecuentes que CBOW, que promedia el contexto y diluye la señal.
- **`min_count=5`:** filtra palabras que aparecen menos de 5 veces. Con un corpus tan pequeño, esas palabras (nombres propios secundarios que solo aparecen 1-2 veces, errores de tokenización, etc.) no tienen contexto suficiente para estimar un embedding confiable y solo agregan ruido. Bajarlo dejaría vectores muy pobres; subirlo mucho (p. ej. 20) tiraría personajes secundarios relevantes (`eumaeus` aparece 83 veces, pero otros nombres propios del texto son bastante menos frecuentes).
- **`window=5`:** una ventana más amplia que la mínima (2-3) prioriza capturar relación **temática/semántica** (de qué se habla alrededor de la palabra) por sobre relación puramente **sintáctica** (qué palabra va justo al lado). Para el objetivo del desafío (encontrar términos semánticamente relacionados y clusters temáticos) conviene una ventana amplia.
- **`vector_size=100`:** el notebook original (con un corpus de letras de canciones, más grande) usaba 300 dimensiones. Con un vocabulario de ~2.000 palabras y ~3.580 oraciones de entrenamiento, 300 dimensiones son demasiados parámetros libres por palabra para la cantidad de datos disponible (sobreajuste / dimensiones mal estimadas). 100 es un tamaño más conservador y razonable para este volumen de corpus.
- **`negative=20`:** cantidad de negative samples por cada muestra positiva. Valores más altos (10-20, en vez del default de 5) suelen ayudar cuando hay pocos datos, porque cada actualización de entrenamiento aprovecha más contraste por muestra positiva, compensando la escasez de ejemplos.
- **`epochs=20`** (en el `.train()`): el default de Gensim es 5 épocas, pensado para corpus grandes donde cada época ya aporta muchísimas actualizaciones. Con un corpus pequeño como este, hacen falta más pasadas para que el modelo vea suficientes veces cada palabra; la curva de loss (ver más abajo) sigue bajando de forma sostenida hasta cerca de la época 15-16, lo que sugiere que menos épocas hubiera dejado el entrenamiento incompleto.


In [ ]:
w2v_model = Word2Vec(
    min_count=5,  # frecuencia mínima de palabra para incluirla en el vocabulario
    window=5,  # cant de palabras antes y desp de la predicha
    vector_size=100,  # dimensionalidad de los vectores
    negative=20,  # cantidad de negative samples
    workers=1,  # reproducibilidad (con más workers el entrenamiento deja de ser determinista)
    seed=42,
    sg=1,  # modelo 0:CBOW  1:skipgram
)

w2v_model.build_vocab(sentences)

print("Cantidad de docs (oraciones) en el corpus:", w2v_model.corpus_count)
print("Cantidad de words distintas en el vocabulario:", len(w2v_model.wv.index_to_key))

In [ ]:
w2v_model.train(
    sentences,
    total_examples=w2v_model.corpus_count,
    epochs=20,
    compute_loss=True,
    callbacks=[LossLogger()],
)

## Términos de interés: palabras más y menos similares

Elegimos tres términos relevantes para _La Odisea_: `ulysses` (protagonista), `minerva` (diosa que lo protege) y `suitors` (los pretendientes de Penélope, antagonistas centrales de la trama).

Para las "menos similares" usamos el mismo truco que Gensim permite vía `most_similar(negative=[...])`: en lugar de buscar el vector más parecido, busca el más opuesto en el espacio de embeddings.


In [ ]:
# Palabras que MÁS se relacionan con "ulysses":
w2v_model.wv.most_similar(positive=["ulysses"], topn=10)

In [ ]:
# Palabras que MENOS se relacionan con "ulysses":
w2v_model.wv.most_similar(negative=["ulysses"], topn=10)

**Interpretación:** las palabras más similares a `ulysses` son casi todas nombres propios de otros personajes (`theoclymenus`, `agelaus`, `amphimedon`, `amphinomus`, `laodamas`, `euryalus`) más algunos verbos de reconocimiento (`recognised`, `bethought`) — tiene sentido: "Ulysses" aparece en contextos narrativos muy similares a los de otros personajes (sujeto de verbos de acción/diálogo), que es justo el tipo de similitud "distribucional" que capta Word2Vec. Las "menos similares" son sustantivos concretos y cuantificadores genéricos (`twelve`, `wheat`, `island`, `cattle`, `herds`) sin ninguna relación temática clara con el protagonista; como se explicó arriba, esto es más ruido que señal.


In [ ]:
# Palabras que MÁS se relacionan con "minerva":
w2v_model.wv.most_similar(positive=["minerva"], topn=10)

In [ ]:
# Palabras que MENOS se relacionan con "minerva":
w2v_model.wv.most_similar(negative=["minerva"], topn=10)

**Interpretación:** junto a `minerva` aparecen `helen` y `arete` (otros personajes femeninos importantes) y palabras ligadas a su rol narrativo como diosa que interviene disfrazada u orienta a los mortales (`likeness`, `vision`, `bethought`). Tiene sentido temático: Minerva suele aparecer "tomando la apariencia de" alguien o inspirando pensamientos a los personajes. Las "menos similares" vuelven a ser sustantivos/adjetivos sin relación clara (`wicked`, `age`, `table`, `herds`), reforzando que ese lado de la consulta aporta poca información interpretable.


In [ ]:
# Palabras que MÁS se relacionan con "suitors":
w2v_model.wv.most_similar(positive=["suitors"], topn=10)

In [ ]:
# Palabras que MENOS se relacionan con "suitors":
w2v_model.wv.most_similar(negative=["suitors"], topn=10)

**Interpretación:** `suitors` aparece asociado a `murder`, `plot`, `antinous` (uno de los pretendientes nombrados en el texto) y `judgement` — consistente con la trama: los pretendientes conspiran (`plot`), son asesinados (`murder`) al final por Ulises, y el relato pasa juicio sobre su comportamiento (`judgement`, `wasting`, por los recursos de la casa de Ulises que consumen). Es el cluster más claramente narrativo de los tres. Las "menos similares" (`fair`, `lies`, `trees`, `darkness`) no tienen una lectura temática evidente.


## Reducción de dimensionalidad y visualización

Reducimos los vectores a 2D con t-SNE. Una primera prueba tomando las `MAX_WORDS` palabras más frecuentes del vocabulario dio un gráfico dominado por _stopwords_ (pronombres, artículos, auxiliares) — se agrupan por rol gramatical, pero no dicen mucho del contenido del libro. Para que la visualización sea más informativa, filtramos las stopwords (con la lista de `gensim.parsing.preprocessing`) antes de quedarnos con las `MAX_WORDS` palabras de contenido más frecuentes.


In [ ]:
import numpy as np
from gensim.parsing.preprocessing import STOPWORDS
from sklearn.manifold import TSNE

MAX_WORDS = 150

# index_to_key ya viene ordenado por frecuencia descendente
content_words = [w for w in w2v_model.wv.index_to_key if w not in STOPWORDS][:MAX_WORDS]
vectors = np.asarray([w2v_model.wv[w] for w in content_words])

tsne = TSNE(n_components=2, random_state=42, init="pca", perplexity=30)
vecs_2d = tsne.fit_transform(vectors)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 12))
plt.scatter(vecs_2d[:, 0], vecs_2d[:, 1], alpha=0.6)
for i, word in enumerate(content_words):
    plt.annotate(word, (vecs_2d[i, 0], vecs_2d[i, 1]), fontsize=9, alpha=0.85)
plt.title(f"Proyección 2D (t-SNE) de los {MAX_WORDS} términos de contenido más frecuentes")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.tight_layout()
plt.show()

**Interpretación de los grupos observados:**

- **Dioses y realeza** (`jove`, `neptune`, `minerva`, `king`, `alcinous`) aparecen agrupados en un extremo del gráfico, junto a otro cluster cercano de vocabulario "celestial" (`sun`, `god`, `gods`, `heaven`) — el modelo separa razonablemente el campo semántico de lo divino del resto del texto.
- **Personajes centrales de la trama** (`penelope`, `telemachus`, `ulysses`, `eumaeus`, `menelaus`, `suitors`, `strangers`) forman un cluster compacto y bien diferenciado, lo cual tiene sentido: comparten contextos narrativos muy similares (sujetos/objetos de los mismos verbos de diálogo y acción).
- **Vocabulario náutico** (`sea`, `ship`, `ships`, `island`, `land`) queda agrupado y cercano a **topónimos** (`ithaca`, `country`, `city`, `town`, `troy`, `achaeans`, `phaeacians`) — coherente con que la Odisea es, literalmente, un relato de viaje entre distintos lugares por mar.
- **Vínculos familiares** (`husband`, `mother`, `father`, `daughter`, `wife`, `son`) se agrupan entre sí, separados del resto — un campo semántico muy consistente y fácil de detectar incluso con un corpus pequeño.
- **Tiempo/rutina diaria** (`time`, `day`, `night`, `morning`, `sleep`) y **hospitalidad** (`wine`, `drink`, `eat`) forman sus propios subgrupos, reflejando motivos recurrentes del poema (banquetes, el paso de los días).
- Curiosamente, aparece un pequeño grupo aislado con `book`, `odyssey`, `writer`, `greek` — probablemente proveniente del prefacio del traductor (Samuel Butler) que se conservó en el texto y no del poema en sí; es un buen ejemplo de cómo el "ruido" (prefacios, notas) puede colarse como su propio cluster.

En general, los clusters reflejan bien los campos semánticos del libro (personajes, lugares, familia, lo divino, rutina) a pesar de tratarse de un corpus relativamente pequeño (~3.580 oraciones, ~2.000 palabras con frecuencia ≥ 5).
